In [ ]:

import numpy as np

from sklearn.datasets import make_friedman1
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error


# ==========================================
# STEP 1: GENERATE DATA
# ==========================================

def make_data(seed=0):

    X, y = make_friedman1(
        n_samples=600,
        noise=3.0,
        random_state=seed
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=seed
    )

    return X_train, X_test, y_train, y_test


# ==========================================
# STEP 2: STAGED TEST ERRORS
# ==========================================

def staged_test_errors(
    model,
    X_train,
    y_train,
    X_test,
    y_test
):

    model.fit(X_train, y_train)

    errors = []

    for predictions in model.staged_predict(X_test):

        mse = mean_squared_error(
            y_test,
            predictions
        )

        errors.append(mse)

    return np.array(errors)


# ==========================================
# STEP 3: BEST NUMBER OF ESTIMATORS
# ==========================================

def best_n_estimators(errors):

    return int(np.argmin(errors) + 1)


# ==========================================
# STEP 4: OVERFITTING GAP
# ==========================================

def overfit_gap(errors):

    errors = np.asarray(errors)

    return float(errors[-1] - np.min(errors))


# ==========================================
# STEP 5: DETECT OVERFITTING
# ==========================================

def is_overfitting(errors, tol=1e-9):

    errors = np.asarray(errors)

    return bool(
        errors[-1] > np.min(errors) + tol
    )


# ==========================================
# STEP 6: VERIFY ERROR FUNCTIONS
# ==========================================

errs = np.array([
    10.0,
    6.0,
    4.0,
    3.0,
    3.5,
    5.0
])

print("Best Number of Estimators:")
print(best_n_estimators(errs))

# Expected: 4


print("\nOverfitting Gap:")
print(overfit_gap(errs))

# Expected: 2.0


print("\nIs Overfitting?")
print(is_overfitting(errs))

# Expected: True


# ==========================================
# STEP 7: TEST NON-OVERFITTING CASE
# ==========================================

decreasing_errors = np.array([
    10.0,
    8.0,
    6.0,
    4.0,
    2.0
])

print("\nNon-Overfitting Test:")
print(is_overfitting(decreasing_errors))

# Expected: False


# ==========================================
# STEP 8: GENERATE TRAIN AND TEST DATA
# ==========================================

X_train, X_test, y_train, y_test = make_data(0)

print("\nDataset Shapes:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


# ==========================================
# STEP 9: CREATE GRADIENT BOOSTING MODEL
# ==========================================

model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.3,
    max_depth=5,
    random_state=0
)


# ==========================================
# STEP 10: CALCULATE STAGED TEST ERRORS
# ==========================================

errors = staged_test_errors(
    model,
    X_train,
    y_train,
    X_test,
    y_test
)

print("\nStaged Test Errors Shape:")
print(errors.shape)

print("\nFirst 10 Test Errors:")
print(errors[:10].round(3))


# ==========================================
# STEP 11: FIND OPTIMAL ESTIMATORS
# ==========================================

best_k = best_n_estimators(errors)

print("\nBest Number of Estimators:")
print(best_k)


# ==========================================
# STEP 12: CHECK OVERFITTING
# ==========================================

gap = overfit_gap(errors)

print("\nOverfitting Gap:")
print(round(gap, 3))

print("\nIs Model Overfitting?")
print(is_overfitting(errors))


# ==========================================
# STEP 13: VERIFY EARLY STOPPING
# ==========================================

print("\nEarly Stopping Verification:")

print("Total Estimators:", model.n_estimators)
print("Best Estimators:", best_k)

print("Best Test MSE:", round(errors[best_k - 1], 3))
print("Final Test MSE:", round(errors[-1], 3))

print("Best Stage Before Final Stage:", best_k < 200)
print("Overfitting Gap Positive:", gap > 0)
